In [ ]:
#Imports
import numpy as np
import time #Only used for in the testing cell below
#Define helper Functions

#Helper for Algorithm A
def isValid(coloring, k):
    '''
    isValid takes a coloring and an integer k and returns true if the coloring contains no monochromatic arithmetic progressions of length k.
    '''
    n = len(coloring) - 1
    d = 1
    #bound = max distance for an AP of length k from the provided coloring
    bound = n/(k - 1)
    #Iterate over each possible value for distance between terms of an AP
    while(d <= bound):
        #Check if AP with current d is possible. If not, increment and continue
        if((k - 1) * d > n):
            d += 1
            continue
        #Create iterator/id for the following loop
        i = -1
        #Set flag to check if AP exists
        flag = True
        #Iterate over the k-1 terms that could be in an AP with the last term
        for _ in range(k - 1):
            #increment id
            i = i - d
            #Validate array bound
            if(i < -n - 1):
                flag = False
                break
            #Check if next term is monochrome
            if(coloring[-1] != coloring[i]):
                #if the coloring isn't consistent then the AP isn't valid
                flag = False
                break
        #If a valid AP was found, return False (as the coloring is valid only if no monochrome AP exists).
        if(flag):
            #print("AP is valid, coloring: ",coloring)
            return False
        #Increment d and loop
        d += 1
    #If we reach the max value for d and no monochrome AP is found, then the coloring is valid
    return True

#Helper for Algorithm B
def prog(c, k, i, j):
    '''
    When called, prog(c, k, i, j) is a function which returns d, the
    maximum difference such that an arithmetic progression with difference d and
    length k terminating at position i is monochromatic of color j, if there is
    no such progression, return -1 instead.
    '''
    #Check if the terminating position is of color j, if not then return -1.
    if(c[i] != j):
        return -1
    #Calculate the maximum possbile value for d 
    bound = (i - 1)/(k - 1) + 1
    #Iterate over all possible values of d, starting with the largest
    for d in range(int(bound), 0, -1):
        #set flag
        valid_ap = True
        for m in range(1, k): #k-1 is the max permissible length of a monochrome AP. We check k-1 terms after the terminating position, thus checking for APs of length n (1 + max permissible length).
            #Validate bounds
            if((i - m*d) < 0):
                valid_ap = False
                break
            #Check color of next term in AP
            if(c[i - m*d] != j):
                valid_ap = False
                break
        if(valid_ap):
            return d
    #If we iterate over all possible values of d and do not find a match, return -1
    return -1

#Define Algorithm A (a python implementation of the algorithm presented in Computer-Generated Van Der Waerden Partitions by R. S. Stevens and R. Shantaram published in 1978).
def algA(k,r):
    '''
    algA takes integers k and r as input and returns the Van Der Waerden number w(k;r). w(k;r) is the shortest length of an r-coloring that is guarenteed to contain a monochromatic arithmetic progression of length k.
    '''
    step = 1
    while(True):
        match step:
            case 1:
                #i represents the current index we are examining
                i = 0
                #c is the coloring stored as an array of integers from [0,k-1]
                c = [0]
                #imax represents the highest index checked
                imax = 0
                #advance to step 2
                step = 2
            case 2:
                #increment i
                i += 1
                #color c[i] with 0
                c.append(0)
                #advance to step 3
                step = 3
            case 3:
                #if c is a valid coloring (no monochromatic APs of length k exist) jump to step 2. Otherwise advance to step 4
                if(isValid(c, k)):
                    step = 2
                else:
                    step = 4
            case 4:
                #If c[i] is the last color, go to step 6. Otherwise advance to step 5
                if(c[i] == r - 1):
                    step = 6
                else:
                    step = 5
            case 5:
                #increment the color of c[i], then return to step 3
                c[i] += 1
                step = 3 
            case 6:
                #update imax if needed, then advance to step 7
                if(i > imax):
                    imax = i
                step = 7
            case 7:
                #decrement i and remove the color from c that was associated with the old value of i. Then advance to step 8
                i -= 1
                del c[-1]
                step = 8
            case 8:
                #if i = 0 terminate the algorithm, otherwise return to step 4
                if(i > 0):
                    step = 4    
                else:
                    #Terminate
                    step = 9
            case 9:
                #Terminate the algorithm and return the Van Der Waerden number w(k;r).
                return(imax + 1)

#Define Algorithm B (a python implementation of Algorithm B from Some New Van Der Waerden Numbers by Michael Beeler and Patrick O'Neil published in 1979).
def algB(k, r):
    '''
    algB takes integers k and r as input and returns a tuple containing the Van Der Waerden number w(k;r) and an array containing all of the surviving strings of length w(k;r) - 1 with no monochromatic arithmetic progressions of length k. w(k;r) is the shortest length of an r-coloring that is guarenteed to contain a monochromatic arithmetic progression of length k.
    '''
    step = 1
    c = []
    not_eliminated = []
    d_min = []
    while(True):
        match step:
            case 1:
                MAXL = 0
                i = 0
                recorded_strings = []
                step = 2
                
            case 2:
                #Check if i is a valid index, if not then entries add up to and including it.
                not_eliminated.append(list(range(r)))
                d_min.append(float('inf'))
                #Initialize D_min
                step = 3

            case 3:
                if(len(not_eliminated[i]) < 1):
                    j = -1
                    step = 5
                else:
                    #Complete J = COLOR(I) instruction
                    j = not_eliminated[i][0]
                    c.append(j)
                    del not_eliminated[i][0]
                    D = prog(c, k, i, j)
                    if(D == -1):
                        step = 4
                    else:
                        d_min[i] = min(D, d_min[i])
                        del c[-1]
                        step = 3

            case 4:
                if(i > MAXL):
                    MAXL = i
                    recorded_strings = [(c.copy())]
                else:
                    if(i == MAXL):
                        recorded_strings.append(c.copy())
                i += 1
                step = 2

            case 5:
                #Ensure that we are backing up a minimum of one position
                if (i < len(d_min)):
                    current_d_min = d_min[i] 
                else:
                    current_d_min = max(1, len(c))

                for _ in range(0, current_d_min):
                    if(len(c) > 0):
                        del c[-1]

                i = i-current_d_min

                if(i <= -1):
                    return(MAXL + 2, recorded_strings)
                
                #Delete old list of potential colors and d_min value when backing up
                not_eliminated = not_eliminated[:i+1]
                d_min = d_min[:i+1]

                if(i >= 0):
                    d_min[i] = min(d_min[i], current_d_min)
                step = 3

Run the above cell once to load the imports and algorithms. Afterwards run the cell below to view a time comparison between the algorithms for w(3;2). Feel free to edit the cell to test the algorithms on other values as well.

In [ ]:
#Set parameters
k = 3 #Length of guarenteed arithmetic progression.
r = 3 #Number of colors

#Run Algorithm A
t0 = time.time()
A33 = algA(k,r)
t1 = time.time()
print(f"Algorithm A: w({k};{r}) = {A33} calculated in {t1-t0} seconds.")

#Run Algorithm B
t0 = time.time()
B33 = algB(k,r)[0]
t1 = time.time()
print(f"Algorithm B: w({k};{r}) = {B33} calculated in {t1-t0} seconds.")

Algorithm A: w(3;3) = 27 calculated in 1.595229148864746 seconds.
Algorithm B: w(3;3) = 27 calculated in 0.9874567985534668 seconds.
